<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 10 · Process Zoo and Simulation Diagnostics

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/10_dxquant_processes_zoo.ipynb)


## Notebook Goals

This notebook streamlines process-level experiments in DXQuant. You will

- simulate terminal distributions for multiple models,
- compare simple distribution diagnostics (mean, std, quantiles), and
- understand what changes when you add stochastic volatility or jumps.

The focus is on simulation behavior rather than valuation wrappers.

## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

ENABLE_PLOTS = False
plt = None
if ENABLE_PLOTS:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        plt = None

## 1. Helpers

We define small helpers for diagnostics and for reusing a common time grid.

In [ ]:
def describe(x: np.ndarray, *, name: str) -> None:
    x = np.asarray(x, dtype=float)
    q = np.percentile(x, [1, 5, 50, 95, 99])
    msg = (
        f"{name:>10s}: "
        f"mean={float(np.mean(x)):.4f}  "
        f"std={float(np.std(x, ddof=1)):.4f}  "
        f"p50={float(q[2]):.4f}  "
        f"p95={float(q[3]):.4f}"
    )
    print(msg)

spot = 100.0
maturity = 1.0
steps = 96
paths = 80_000

grid = dx.uniform_time_grid(maturity=maturity, num_steps=steps)
grid[:5], grid[-1]

## 2. GBM (Baseline)

GBM is the classical log-normal diffusion. It is a good baseline for sanity checks and for
isolating numerical issues.


In [ ]:
rate = 0.02
vol = 0.20

gbm = dx.GeometricBrownianMotion(drift=rate, volatility=vol, seed=123)
gbm_paths = gbm.simulate_paths(spot=spot, time_grid=grid, paths=paths)
gbm_terminal = gbm_paths[:, -1]

describe(gbm_terminal, name="GBM")

## 3. Heston (Stochastic Volatility)

Heston couples the spot to a mean-reverting variance process. Compared to GBM, tails and
skew can change materially.


In [ ]:
heston = dx.HestonProcess(
    kappa=2.0,
    theta=0.04,
    sigma=0.6,
    rho=-0.5,
    drift=rate,
    seed=123,
)

spot_paths, var_paths = heston.simulate_paths(
    spot=spot,
    variance=heston.theta,
    time_grid=grid,
    paths=paths,
)

heston_terminal = spot_paths[:, -1]
describe(heston_terminal, name="Heston")

print("variance mean:", float(np.mean(var_paths[:, -1])))

## 4. Merton Jump Diffusion

A jump diffusion adds Poisson-driven jumps to the diffusion. This introduces fatter tails and
rare, large moves.


In [ ]:
jump = dx.JumpDiffusionProcess(
    drift=rate,
    volatility=vol,
    jump_intensity=0.6,
    jump_mean=-0.10,
    jump_std=0.20,
    seed=123,
)

jump_paths = jump.simulate_paths(spot=spot, time_grid=grid, paths=paths)
jump_terminal = jump_paths[:, -1]

describe(jump_terminal, name="Jumps")

## 5. SABR (State-Dependent Volatility)

The SABR process here is a simple stochastic-volatility style model often used as a smile
generator. We simulate the underlying level to compare terminal behavior.


In [ ]:
sabr = dx.SABRProcess(
    initial_value=spot,
    alpha=0.20,
    beta=0.50,
    rho=-0.2,
    vol_vol=0.60,
    drift=rate,
    seed=123,
)

sabr_paths = sabr.simulate_paths(maturity=maturity, steps=steps, paths=paths)
sabr_terminal = sabr_paths[:, -1]

describe(sabr_terminal, name="SABR")

## 6. Optional Visualization

If you enable plots, you can inspect histograms of terminal values.

In [ ]:
if plt is not None:
    series = {
        "GBM": gbm_terminal,
        "Heston": heston_terminal,
        "Jumps": jump_terminal,
        "SABR": sabr_terminal,
    }
    plt.figure(figsize=(10, 5))
    for key, values in series.items():
        plt.hist(values, bins=80, density=True, alpha=0.35, label=key)
    plt.title("Terminal distributions")
    plt.xlabel("terminal value")
    plt.ylabel("density")
    plt.legend()
    plt.show()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">